In [2]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

TRAIN_DIR = r'D:\archive\train'

CLASSES = [
    'Acne and Rosacea Photos',
    'Eczema Photos',
    'Melanoma Skin Cancer Nevi and Moles',
    'Psoriasis pictures Lichen Planus and related diseases'
]

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    brightness_range=[0.85, 1.15],
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical',
    classes=CLASSES,
    subset='training',
    shuffle=True
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical',
    classes=CLASSES,
    subset='validation',
    shuffle=False
)

print("Class indices:", train_generator.class_indices)
print("Training samples:", train_generator.samples)
print("Validation samples:", val_generator.samples)

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

for layer in base_model.layers[:-80]:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(4, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)
lr_reduce = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)

print("Starting training...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[early_stop, lr_reduce]
)

model.save(r'C:\Users\Deepika\Downloads\Skin_disease_classification-main\Skin_disease_classification-main\model\Best_skin_disease_Acc98.h5')
print("Model saved successfully!")
print("Final class order:", list(train_generator.class_indices.keys()))

Found 3155 images belonging to 4 classes.
Found 788 images belonging to 4 classes.
Class indices: {'Acne and Rosacea Photos': 0, 'Eczema Photos': 1, 'Melanoma Skin Cancer Nevi and Moles': 2, 'Psoriasis pictures Lichen Planus and related diseases': 3}
Training samples: 3155
Validation samples: 788
Starting training...
Epoch 1/20
198/198 ━━━━━━━━━━━━━━━━━━━━ 254s 1s/step - accuracy: 0.5312 - loss: 1.0372 - val_accuracy: 0.4759 - val_loss: 1.3359 - learning_rate: 1.0000e-04
Epoch 2/20
198/198 ━━━━━━━━━━━━━━━━━━━━ 182s 752ms/step - accuracy: 0.6799 - loss: 0.7679 - val_accuracy: 0.5152 - val_loss: 1.3319 - learning_rate: 1.0000e-04
Epoch 3/20
198/198 ━━━━━━━━━━━━━━━━━━━━ 187s 677ms/step - accuracy: 0.7537 - loss: 0.6258 - val_accuracy: 0.4429 - val_loss: 1.8939 - learning_rate: 1.0000e-04
Epoch 4/20
198/198 ━━━━━━━━━━━━━━━━━━━━ 132s 669ms/step - accuracy: 0.7883 - loss: 0.5395 - val_accuracy: 0.5203 - val_loss: 1.3824 - learning_rate: 1.0000e-04
Epoch 5/20
198/198 ━━━━━━━━━━━━━━━━━━━━ 143s

Model saved successfully!
Final class order: ['Acne and Rosacea Photos', 'Eczema Photos', 'Melanoma Skin Cancer Nevi and Moles', 'Psoriasis pictures Lichen Planus and related diseases']


In [3]:
import os
path = r'C:\Users\Deepika\Downloads\Skin_disease_classification-main\Skin_disease_classification-main\model\Best_skin_disease_Acc98.h5'
print("File exists:", os.path.exists(path))
print("File size:", os.path.getsize(path) if os.path.exists(path) else "NOT FOUND")

File exists: True
File size: 30138448


In [4]:
model.save(r'C:\Users\Deepika\Downloads\Skin_disease_classification-main\Skin_disease_classification-main\model\Best_skin_disease_Acc98.h5', overwrite=True)
print("Saved!")

import os
size = os.path.getsize(r'C:\Users\Deepika\Downloads\Skin_disease_classification-main\Skin_disease_classification-main\model\Best_skin_disease_Acc98.h5')
print(f"File size: {size / 1024 / 1024:.2f} MB")

Saved!
File size: 28.74 MB


In [5]:
print(model.summary())

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1 (Conv2D)                │ (None, 112, 112, 32)      │             864 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_Conv1 (BatchNormalization) │ (None, 112, 112, 32)      │             128 │ Conv1[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1_relu (ReLU)             │ (None, 112, 112, 32)      │               0 │ bn_Conv1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise       │ (None, 112, 112, 32)      │             288 │ Conv1_relu[0][0]           │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_BN    │ (None, 112, 112, 32)      │             128 │ expanded_conv_depthwise[0… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_relu  │ (None, 112, 112, 32)      │               0 │ expanded_conv_depthwise_B… │
│ (ReLU)                        │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project         │ (None, 112, 112, 16)      │             512 │ expanded_conv_depthwise_r… │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project_BN      │ (None, 112, 112, 16)      │              64 │ expanded_conv_project[0][… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand (Conv2D)       │ (None, 112, 112, 96)      │           1,536 │ expanded_conv_project_BN[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_BN             │ (None, 112, 112, 96)      │             384 │ block_1_expand[0][0]       │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_relu (ReLU)    │ (None, 112, 112, 96)      │               0 │ block_1_expand_BN[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_pad (ZeroPadding2D)   │ (None, 113, 113, 96)      │               0 │ block_1_expand_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_depthwise             │ (None, 56, 56, 96)        │             864 │ block_1_pad[0][0]          │
│ (DepthwiseConv2D)             │                           │               

 Total params: 7,380,302 (28.15 MB)

 Trainable params: 2,396,676 (9.14 MB)

 Non-trainable params: 190,272 (743.25 KB)

 Optimizer params: 4,793,354 (18.29 MB)

None


In [6]:
model.save(r'C:\Users\Deepika\Downloads\Skin_disease_classification-main\Skin_disease_classification-main\model\skin_model.keras')
print("Saved!")

Saved!


In [7]:
# Save in new keras format
model.save(r'C:\Users\Deepika\Downloads\Skin_disease_classification-main\Skin_disease_classification-main\model\skin_model.keras')
print("Saved in keras format!")

import os
size = os.path.getsize(r'C:\Users\Deepika\Downloads\Skin_disease_classification-main\Skin_disease_classification-main\model\skin_model.keras')
print(f"File size: {size / 1024 / 1024:.2f} MB")

Saved in keras format!
File size: 28.77 MB


In [8]:
# Save weights separately
model.save_weights(r'C:\Users\Deepika\Downloads\Skin_disease_classification-main\Skin_disease_classification-main\model\skin_weights.weights.h5')
print("Weights saved!")

Weights saved!


In [9]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model

# Rebuild same architecture
base_model = MobileNetV2(weights=None, include_top=False, input_shape=(224, 224, 3))
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(4, activation='softmax')(x)
new_model = Model(inputs=base_model.input, outputs=predictions)

# Load saved weights
new_model.load_weights(r'C:\Users\Deepika\Downloads\Skin_disease_classification-main\Skin_disease_classification-main\model\skin_weights.weights.h5')
print("✅ Weights loaded successfully!")

# Save as h5
new_model.save(r'C:\Users\Deepika\Downloads\Skin_disease_classification-main\Skin_disease_classification-main\model\skin_model_final.h5')
print("✅ Final model saved!")

✅ Weights loaded successfully!
✅ Final model saved!


In [10]:
import keras
print("Keras version:", keras.__version__)
import tensorflow as tf
print("TF version:", tf.__version__)

Keras version: 3.13.2
TF version: 2.20.0


In [11]:
import tensorflow as tf

# Save in TensorFlow SavedModel format - works across versions
tf.saved_model.save(model, r'C:\Users\Deepika\Downloads\Skin_disease_classification-main\Skin_disease_classification-main\model\saved_model')
print("✅ Saved in SavedModel format!")

TypeError: this __dict__ descriptor does not support '_DictWrapper' objects

In [12]:
# Simple weights only save
model.save_weights(r'C:\Users\Deepika\Downloads\Skin_disease_classification-main\Skin_disease_classification-main\model\weights_only.weights.h5')
print("Weights saved!")

Weights saved!


In [14]:
img_path = r'C:\Users\Deepika\Downloads\flower.jpeg'  # put your flower image path
from PIL import Image
import numpy as np

img = Image.open(img_path).convert("RGB")
img_array = np.array(img.resize((224, 224)))

# Check average skin tone - skin pixels are reddish
r, g, b = img_array[:,:,0].mean(), img_array[:,:,1].mean(), img_array[:,:,2].mean()
print(f"R:{r:.1f} G:{g:.1f} B:{b:.1f}")

preds = model.predict(np.expand_dims(img_array/255.0, axis=0))
print(f"Confidence: {np.max(preds)*100:.2f}%")

R:141.1 G:69.6 B:78.3
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
Confidence: 99.81%


In [15]:
import os
import urllib.request

# Create a not_skin folder with random images
not_skin_dir = r'D:\archive\train\Not Skin'
os.makedirs(not_skin_dir, exist_ok=True)
print("Created Not Skin folder at:", not_skin_dir)
print("Now manually copy 100+ random non-skin images (flowers, cars, food etc) into:")
print(not_skin_dir)

Created Not Skin folder at: D:\archive\train\Not Skin
Now manually copy 100+ random non-skin images (flowers, cars, food etc) into:
D:\archive\train\Not Skin
